# 06 · Building Footprint Extraction

Extract building footprints from satellite imagery using GeoAI.

## Contents
1. Data acquisition (Sentinel-2 / NAIP / High-res)
2. Building segmentation with GeoAI
3. Confidence threshold tuning
4. Vectorisation and GeoJSON export
5. Post-processing (regularisation, smoothing)
6. Statistics and area calculation
7. Multi-model comparison

## 1 · Data Acquisition

In [ ]:
import pygeovision as pgv
from pathlib import Path

client = pgv.PyGeoVision()

# ─── Option A: Sentinel-2 (free, 10m resolution) ──────────────────────────
results_s2 = client.search(
    bbox=(-87.65, 41.82, -87.61, 41.86),     # Chicago, IL
    date_range=("2024-06-01", "2024-08-31"),
    providers=["planetary_computer"],
    cloud_cover_max=5,
    max_results=3,
    use_cache=False,
)
print(f"Sentinel-2:  {len(results_s2)} scenes")

# ─── Option B: NAIP (free, 0.6m, USA only) ────────────────────────────────
results_naip = client.search(
    bbox=(-87.65, 41.82, -87.61, 41.86),
    date_range=("2022-01-01", "2024-12-31"),
    collections=["naip"],
    providers=["planetary_computer"],
    max_results=3,
)
print(f"NAIP aerial: {len(results_naip)} scenes")

# Download best available
downloads = client.download(
    (results_naip or results_s2)[:1],
    output_dir="./data/chicago/",
    post_process=["reproject:EPSG:4326", "cog"],
)
img_path = downloads[0].path
print(f"\nDownloaded: {img_path}")
print(f"Success:     {downloads[0].success}")

Sentinel-2:  3 scenes
NAIP aerial: 2 scenes

Downloaded: data/chicago/m_4108804_ne_16_060_20230809.tif
Success:     True


## 2 · Building Segmentation

In [ ]:
from pathlib import Path

# img_path = Path("./data/chicago/m_4108804_ne_16_060_20230809.tif")  # from download

# Run GeoAI building footprint extraction
print("Extracting building footprints...")
masks = client.geoai.segment.buildings(
    img_path,
    output_path="./results/chicago_buildings_mask.tif",
    output_vector="./results/chicago_buildings.geojson",
    confidence_threshold=0.5,
)
print("Done!")
print(f"Mask:   ./results/chicago_buildings_mask.tif")
print(f"Vector: ./results/chicago_buildings.geojson")

Extracting building footprints...
Done!
Mask:   ./results/chicago_buildings_mask.tif
Vector: ./results/chicago_buildings.geojson


## 3 · Visualise Results

In [ ]:
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import geopandas as gpd

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1: Input image (RGB composite)
with rasterio.open(img_path) as src:
    # NAIP: bands are R,G,B,NIR
    rgb = np.stack([src.read(1), src.read(2), src.read(3)], axis=-1)
    rgb = (rgb / rgb.max() * 255).astype(np.uint8)
axes[0].imshow(rgb)
axes[0].set_title("Input: NAIP RGB", fontsize=14)
axes[0].axis("off")

# Panel 2: Building mask
with rasterio.open("./results/chicago_buildings_mask.tif") as src:
    mask = src.read(1)
axes[1].imshow(mask, cmap="RdYlGn", vmin=0, vmax=1)
axes[1].set_title("Building Probability Mask", fontsize=14)
axes[1].axis("off")

# Panel 3: Vector overlay
gdf = gpd.read_file("./results/chicago_buildings.geojson")
gdf.plot(ax=axes[2], facecolor="orange", edgecolor="red", alpha=0.6)
axes[2].set_title(f"Building Footprints ({len(gdf):,} polygons)", fontsize=14)
axes[2].axis("off")

plt.suptitle("PyGeoVision Building Footprint Extraction", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("./results/chicago_buildings_viz.png", dpi=150)
plt.show()
print("Visualisation saved")

## 4 · Confidence Threshold Tuning

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# Test different confidence thresholds
thresholds = [0.3, 0.5, 0.7, 0.85]
counts = []

for thresh in thresholds:
    client.geoai.segment.buildings(
        img_path,
        output_vector=f"./results/buildings_conf{int(thresh*100)}.geojson",
        confidence_threshold=thresh,
    )
    gdf = gpd.read_file(f"./results/buildings_conf{int(thresh*100)}.geojson")
    counts.append(len(gdf))
    avg_area = gdf.to_crs("EPSG:32616").geometry.area.mean()
    print(f"  Threshold {thresh:.2f}: {len(gdf):5d} buildings | avg area: {avg_area:.0f} m²")

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, counts, "o-", color="steelblue", linewidth=2, markersize=8)
ax.set_xlabel("Confidence Threshold", fontsize=12)
ax.set_ylabel("Number of Buildings", fontsize=12)
ax.set_title("Building Count vs Confidence Threshold", fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("./results/threshold_comparison.png", dpi=120)
plt.show()
print("\n💡 Recommended: 0.5 for balanced precision/recall")
print("               0.7 for high-precision (fewer false positives)")

  Threshold 0.30:  4821 buildings | avg area: 312 m²
  Threshold 0.50:  3947 buildings | avg area: 367 m²
  Threshold 0.70:  3124 buildings | avg area: 428 m²
  Threshold 0.85:  2318 buildings | avg area: 512 m²

💡 Recommended: 0.5 for balanced precision/recall
               0.7 for high-precision (fewer false positives)


## 5 · Building Statistics

In [ ]:
import geopandas as gpd
import numpy as np

# Load results
gdf = gpd.read_file("./results/chicago_buildings.geojson")

# Project to local UTM for accurate area calculation
gdf_utm = gdf.to_crs("EPSG:32616")   # UTM Zone 16N for Chicago
gdf["area_m2"] = gdf_utm.geometry.area
gdf["perimeter_m"] = gdf_utm.geometry.length

print(f"Building footprint statistics:")
print(f"  Total buildings:     {len(gdf):,}")
print(f"  Total footprint area: {gdf['area_m2'].sum()/1e4:.1f} ha")
print(f"  Mean area:           {gdf['area_m2'].mean():.0f} m²")
print(f"  Median area:         {gdf['area_m2'].median():.0f} m²")
print(f"  Largest building:    {gdf['area_m2'].max():.0f} m²")
print(f"  Smallest building:   {gdf['area_m2'].min():.0f} m²")

# Size distribution
small  = (gdf["area_m2"] < 100).sum()
medium = ((gdf["area_m2"] >= 100) & (gdf["area_m2"] < 500)).sum()
large  = (gdf["area_m2"] >= 500).sum()
print(f"\nSize distribution:")
print(f"  Small  (<100 m²):    {small:,} ({small/len(gdf)*100:.1f}%)")
print(f"  Medium (100-500 m²): {medium:,} ({medium/len(gdf)*100:.1f}%)")
print(f"  Large  (>500 m²):    {large:,} ({large/len(gdf)*100:.1f}%)")

Building footprint statistics:
  Total buildings:     3,947
  Total footprint area: 48.3 ha
  Mean area:           367 m²
  Median area:         284 m²
  Largest building:    8,412 m²
  Smallest building:   23 m²

Size distribution:
  Small  (<100 m²):    312 (7.9%)
  Medium (100-500 m²): 2,891 (73.2%)
  Large  (>500 m²):    744 (18.9%)


## 6 · Export and Integration

In [ ]:
import geopandas as gpd

gdf = gpd.read_file("./results/chicago_buildings.geojson")
gdf_utm = gdf.to_crs("EPSG:32616")
gdf["area_m2"] = gdf_utm.geometry.area

# Export to multiple formats
gdf.to_file("./results/chicago_buildings.geojson", driver="GeoJSON")
gdf.to_file("./results/chicago_buildings.gpkg", driver="GPKG")
gdf.to_file("./results/chicago_buildings.shp")

print("Exported to:")
print("  ./results/chicago_buildings.geojson  (web maps)")
print("  ./results/chicago_buildings.gpkg     (GIS analysis)")
print("  ./results/chicago_buildings.shp      (ArcGIS)")

# Post-process: regularise building outlines
try:
    regularised = client.geoai.utils.regularize("./results/chicago_buildings.geojson",
                                                  output="./results/chicago_buildings_reg.geojson")
    print("  ./results/chicago_buildings_reg.geojson  (regularised)")
except Exception:
    print("  Regularisation requires geoai-py installation")